FAISS (Facebook AI Similarity Search) is a vector database library used to:FAISS is a high-performance local vector search engine, ideal for fast similarity search in GenAI systems. 

Store embeddings (vectors) and quickly find the most similar ones.
In GenAI systems, FAISS helps answer questions like:
“Which document chunk is most similar to this question?”
“Which past conversation is closest to this query?”



In [25]:
import sys
!{sys.executable} -m pip install -r requirements.txt

In [9]:
# Import a document loader that reads plain text files
# TextLoader converts a .txt file into LangChain Document objects
from langchain_community.document_loaders import TextLoader

# Import a text splitter
# CharacterTextSplitter breaks long text into smaller overlapping chunks
from langchain_text_splitters import CharacterTextSplitter

# Import Ollama embeddings
# OllamaEmbeddings generates vector embeddings using a locally running Ollama model
from langchain_community.embeddings import OllamaEmbeddings

# Import FAISS vector store
# FAISS is used to store and efficiently search embeddings (similarity search)
from langchain_community.vectorstores import FAISS


# ---------------------------------------------------
# Step 1: Split text into manageable chunks
# ---------------------------------------------------

# Create a text splitter
# chunk_size = 100 characters per chunk
# chunk_overlap = 30 characters overlap between consecutive chunks
# Overlap helps preserve context across chunk boundaries
text_splitter = CharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=30
)

# Load the text file ("stories.txt") and split it into chunks
# .load() → loads text into Document objects
# .split_documents() → splits documents into smaller chunks
docs = text_splitter.split_documents(
    TextLoader("stories.txt").load()
)


# ---------------------------------------------------
# Step 2: Create embeddings using Ollama
# ---------------------------------------------------

# Initialize the embedding model
# "qwen3-embedding:0.6b" is a lightweight embedding model
# It converts text into high-dimensional vectors
embeddings = OllamaEmbeddings(
    model="qwen3-embedding:0.6b"
)


# ---------------------------------------------------
# Step 3: Store embeddings in FAISS vector database
# ---------------------------------------------------

# Create a FAISS vector store from document chunks
# Each chunk is embedded and indexed for fast similarity search
db = FAISS.from_documents(
    docs,
    embeddings
)



Created a chunk of size 202, which is longer than the specified 100
Created a chunk of size 215, which is longer than the specified 100
Created a chunk of size 197, which is longer than the specified 100
Created a chunk of size 201, which is longer than the specified 100


In [10]:
# Define the input query
# This is the natural language text we want to search for semantically
input_query = "Slowly, Arun began to improve. He fixed small bugs"


# Perform similarity search in the FAISS vector store
# 1. The query text is converted into an embedding
# 2. FAISS compares this embedding with stored document embeddings
# 3. It returns the most semantically similar document chunks
inference = db.similarity_search(input_query)


# Access the most relevant result
# inference is a list of Document objects sorted by similarity (best first)
# page_content contains the actual text chunk retrieved from the source file
inference[0].page_content

'Slowly, Arun began to improve. He fixed small bugs, then larger ones. One evening, while debugging an issue others had avoided, he found a simple solution that saved the team several hours of work.'

### Use FAISS as a Retriever

In [ ]:
# Convert the FAISS vector store into a retriever interface
# A retriever is a standardized LangChain abstraction
# It is used by chains (like RetrievalQA, RAG, Agents)
retriever = db.as_retriever()


# Invoke the retriever with the input query
# 1. The query is embedded using the same embedding model
# 2. FAISS performs similarity search over stored vectors
# 3. A list of relevant Document objects is returned
retrieved_docs = retriever.invoke(input_query)


# retrieved_docs is a list of Document objects
# Each Document contains:
# - page_content → the retrieved text chunk
# - metadata → source information (file name, chunk index, etc.)
retrieved_docs

[Document(id='f986918e-06ec-4d4d-90c8-a2dbad555b9e', metadata={'source': 'stories.txt'}, page_content='Slowly, Arun began to improve. He fixed small bugs, then larger ones. One evening, while debugging an issue others had avoided, he found a simple solution that saved the team several hours of work.'),
 Document(id='a7be0525-d216-49db-a679-739b94709c0f', metadata={'source': 'stories.txt'}, page_content='Six months later, Arun led his first feature release. During the team meeting, Meera praised his persistence and growth. Arun realized that success wasn’t about knowing everything at the start—it was about learning consistently and asking for help.'),
 Document(id='89dccc62-281e-4375-95cc-e071ccffd16a', metadata={'source': 'stories.txt'}, page_content='At the office, Arun struggled at first. The codebase was large, the deadlines were tight, and he often doubted himself. His manager, Meera, noticed this and encouraged him to ask questions instead of staying silent.'),
 Document(id='6756c

### Similarity search with score 

In [13]:
# ---------------------------------------------------
# Similarity search WITH scores
# ---------------------------------------------------

# Perform semantic similarity search and also return similarity scores
# Each result is a tuple:
# (Document, score)
#
# - Document → the retrieved text chunk
# - score    → distance between query embedding and document embedding
#              (lower score = more similar)
docs_and_sscore = db.similarity_search_with_score(input_query)


# Inspect the results
docs_and_sscore

[(Document(id='f986918e-06ec-4d4d-90c8-a2dbad555b9e', metadata={'source': 'stories.txt'}, page_content='Slowly, Arun began to improve. He fixed small bugs, then larger ones. One evening, while debugging an issue others had avoided, he found a simple solution that saved the team several hours of work.'),
  np.float32(0.1781106)),
 (Document(id='a7be0525-d216-49db-a679-739b94709c0f', metadata={'source': 'stories.txt'}, page_content='Six months later, Arun led his first feature release. During the team meeting, Meera praised his persistence and growth. Arun realized that success wasn’t about knowing everything at the start—it was about learning consistently and asking for help.'),
  np.float32(0.6167836)),
 (Document(id='89dccc62-281e-4375-95cc-e071ccffd16a', metadata={'source': 'stories.txt'}, page_content='At the office, Arun struggled at first. The codebase was large, the deadlines were tight, and he often doubted himself. His manager, Meera, noticed this and encouraged him to ask ques

In [ ]:
# ---------------------------------------------------
# Similarity search using a precomputed embedding vector
# ---------------------------------------------------

# Step 1: Convert the query text into an embedding vector
# This produces a numerical representation of the query in semantic space
embeddings_vector = embeddings.embed_query(input_query)


# Step 2: Perform similarity search directly using the vector
# similarity_search_by_vector():
# - Skips text embedding (already done)
# - Compares the given vector with stored document vectors
# - Returns the most semantically similar Document objects
results = db.similarity_search_by_vector(embeddings_vector)


# Inspect the retrieved documents
results



[Document(id='f986918e-06ec-4d4d-90c8-a2dbad555b9e', metadata={'source': 'stories.txt'}, page_content='Slowly, Arun began to improve. He fixed small bugs, then larger ones. One evening, while debugging an issue others had avoided, he found a simple solution that saved the team several hours of work.'),
 Document(id='a7be0525-d216-49db-a679-739b94709c0f', metadata={'source': 'stories.txt'}, page_content='Six months later, Arun led his first feature release. During the team meeting, Meera praised his persistence and growth. Arun realized that success wasn’t about knowing everything at the start—it was about learning consistently and asking for help.'),
 Document(id='89dccc62-281e-4375-95cc-e071ccffd16a', metadata={'source': 'stories.txt'}, page_content='At the office, Arun struggled at first. The codebase was large, the deadlines were tight, and he often doubted himself. His manager, Meera, noticed this and encouraged him to ask questions instead of staying silent.'),
 Document(id='6756c

| Method                                | What it expects | When to use              |
| ------------------------------------- | --------------- | ------------------------ |
| `similarity_search(text)`             | Raw text        | Quick experiments        |
| `embed_query()`                       | Text → vector   | Custom pipelines         |
| `similarity_search_by_vector(vector)` | Vector          | Optimized / advanced use |
| `as_retriever()`                      | Text            | Chains & agents          |


In [16]:
### Save in local 
db.save_local("faiss.index")

In [23]:
new_db=FAISS.load_local("Faiss.index",embeddings,allow_dangerous_deserialization=True)
docs=new_db.similarity_search(input_query)
docs[0].page_content

'Slowly, Arun began to improve. He fixed small bugs, then larger ones. One evening, while debugging an issue others had avoided, he found a simple solution that saved the team several hours of work.'